# 04 — Final Results (report-ready)

Compiles the whole project into the figures and tables that go in the report:

1. the **ablation** verdict — does behaviour beat quant-only?
2. a realistic **backtest** equity curve vs buy-and-hold,
3. **SHAP** interpretability — did fear/greed/herd actually drive the calls?

All logic lives in `src/`; this notebook orchestrates and renders. Figures are saved to
`results/figures/`.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))  # so 'import src...' works from notebooks/

import pandas as pd, matplotlib.pyplot as plt

from src.config import load_config
from src.experiments import run_ablation, build_model_frame, feature_columns
from src.data.loader import load_ohlcv
from src.data.splitter import single_split
from src.models import build_model
from src.backtest import run_backtest, buy_and_hold, signal_from_predictions
from src.evaluation import financial_metrics
from src.visualize import (plot_ablation_comparison, plot_equity_curve, plot_drawdown,
                           plot_confusion, plot_feature_importance, shap_summary,
                           behavioral_shap_ranking)

FIG = '../results/figures'
config = load_config('../config.yaml')
config['data']['raw_dir'] = '../data/raw'  # reuse the cached snapshot instead of re-downloading
config['data']['tickers']

## 1. The ablation verdict

Does adding fear/greed/herd beat the quant-only baseline?

In [ ]:
results = run_ablation(config)
summary, significance = results['summary'], results['significance']
summary

In [ ]:
fig = plot_ablation_comparison(summary, metric='accuracy', save_path=f'{FIG}/ablation_accuracy.png')
fig2 = plot_ablation_comparison(summary, metric='sharpe', save_path=f'{FIG}/ablation_sharpe.png')
plt.show()

**Is the gap statistically real?** (paired test across folds)

In [ ]:
significance

## 2. Realistic backtest (every ticker)

Train the behaviour-aware model on the in-sample window for **each** ticker, then trade its
out-of-sample predictions through `src.backtest` (positions, transaction costs) and compare to
buy-and-hold. The resulting table is Section 4 of `RESULTS.md`. We stash each ticker's trained
model and test window in `runs` so the SHAP section below can reuse MSFT without retraining.

In [ ]:
# Realistic backtest for every ticker -> the Section-4 trading table.
runs = {}
rows = []
for ticker in config['data']['tickers']:
    raw = load_ohlcv(ticker, config['data']['start_date'], config['data']['end_date'], config['data']['raw_dir'])
    frame = build_model_frame(raw, config)
    cols = feature_columns(frame, 'quant_plus_behavioral')

    train_idx, test_idx = single_split(frame, config['split'].get('test_size', 0.2))
    model = build_model('xgboost', config).fit(frame.loc[train_idx, cols], frame.loc[train_idx, 'target'])
    pred = model.predict(frame.loc[test_idx, cols])

    test_prices = frame.loc[test_idx, 'Close']
    signals = signal_from_predictions(pred, index=test_idx, allow_short=config['backtest'].get('allow_short', False))
    bt = run_backtest(test_prices, signals,
                      initial_capital=config['backtest']['initial_capital'],
                      transaction_cost=config['backtest']['transaction_cost'])
    bench = buy_and_hold(test_prices, config['backtest']['initial_capital'])

    fin, bench_fin = financial_metrics(bt['equity']), financial_metrics(bench)
    rows.append({
        'ticker': ticker,
        'accuracy': (pred == frame.loc[test_idx, 'target']).mean(),
        'strategy_return': fin['total_return'], 'strategy_sharpe': fin['sharpe'],
        'max_drawdown': fin['max_drawdown'],
        'buy_hold_return': bench_fin['total_return'], 'buy_hold_sharpe': bench_fin['sharpe'],
    })
    runs[ticker] = dict(frame=frame, cols=cols, model=model, pred=pred, test_idx=test_idx, bt=bt, bench=bench)

trading_table = pd.DataFrame(rows).set_index('ticker').round(3)
trading_table

In [ ]:
# Per-ticker report figures: equity vs buy-and-hold, drawdown, confusion, feature importances.
for ticker, r in runs.items():
    plot_equity_curve(r['bt']['equity'], benchmark=r['bench'], save_path=f'{FIG}/{ticker}_equity.png')
    plot_drawdown(r['bt']['equity'], save_path=f'{FIG}/{ticker}_drawdown.png')
    plot_confusion(r['frame'].loc[r['test_idx'], 'target'], r['pred'], save_path=f'{FIG}/{ticker}_confusion.png')
    plot_feature_importance(r['model'].feature_importances(), top_n=15, save_path=f'{FIG}/{ticker}_importance.png')
plt.show()

## 3. SHAP — did behaviour actually drive the decisions?

The payoff question. If fear/greed/herd rank near the top of mean |SHAP|, the model genuinely
leaned on investor psychology — not just price technicals. We focus on **MSFT**, the ticker where
the behavioral signal is strongest (RESULTS.md §5), reusing the model trained above.

In [ ]:
# SHAP on MSFT (xgboost) — the ticker where behavioral features rank highest.
shap_ticker = 'MSFT'
r = runs[shap_ticker]
X_shap = r['frame'].loc[r['test_idx'], r['cols']]

ranking = behavioral_shap_ranking(r['model'], X_shap, max_samples=300)
ranking.head(15)

In [ ]:
shap_summary(r['model'], X_shap, save_path=f'{FIG}/{shap_ticker}_shap_summary.png', max_samples=300)
plt.show()

In [ ]:
# Where the three behavioral signals landed among all features (backs the conclusion):
# greed ranks #1 of all features for MSFT, herd mid-pack, fear near the bottom.
ranking.assign(rank=range(1, len(ranking) + 1)).loc[ranking['is_behavioral'], ['rank', 'mean_abs_shap']]

## Conclusion (real run)

- **Prediction:** behavioral features changed accuracy by only **±0.5pp** (not significant; 1/18 tests at p<0.05, within multiple-comparison noise). Overall accuracy **0.511** daily, **0.540** at a weekly horizon.
- **Trading:** the directional strategy was profitable in absolute terms (Sharpe 0.86–1.57, returns +18–54%) but **underperformed buy-and-hold** on every ticker — accuracy near chance can't beat a rising market.
- **Psychology:** the most influential behavioral signal was **greed** (SHAP rank #1 for MSFT) — the model leans on it, but that usage doesn't generalize into a directional edge.

Ties back to the proposal's expected outcomes: a behaviour-aware strategy, a traditional-vs-behavioral comparison, and insights into the role of behavioral factors. The honest finding — **no significant behavioral edge, robust across horizons** — is the deliverable. Full write-up in `RESULTS.md`.